# YOLOv5 Graffiti Detection - Execution Guide

## ⚠️ IMPORTANT: Run cells in this exact order from top to bottom:

### Phase 1: Setup (MUST RUN FIRST)
1. **Cell 1** - Imports and global constants
2. **Cell 8** - Download YOLOv5 repository + weights ⏱️ ~5-10 minutes first time

### Phase 2: Prepare Data
3. **Cell 2** - Convert CSV annotations to YOLO format
4. **Cell 9** - Create dummy dataset for testing ⏱️ ~5 seconds
5. **Cell 3** - Define prepare_dataset() function

### Phase 3: Define Functions
6. **Cell 4** - Define evaluate_model() function
7. **Cell 5** - Define load_ground_truth() and compute_iou() functions
8. **Cell 6** - Define train_yolo_and_save() function

### Phase 4: Training (MAIN EXECUTION)
9. **Cell 7** - Run iterative training + evaluation ⏱️ ~30-60 minutes

### Phase 5: Analysis & Inference (Optional)
10. **Cell 10** - Organize CSV outputs + pick best samples
11. **Cell 11** - Visualize detections on images
12. **Cell 12** - Process video files with inference

---

## Quick Start:
- Just run from Cell 1 → Cell 7 sequentially
- Cells 10-12 are optional for post-training analysis
- Cell 8 is the slowest (first-time download only)

In [115]:
# Imports and global constants

import os
import random
import shutil
import subprocess
from pathlib import Path

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import yaml
import pandas as pd
import torch
import cv2

# Adjust these to your setup:
BASE     = Path('D:\COS40007\Week_06')
YOLO_DIR = BASE / 'yolov5'

# Device for inference/training
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [116]:
# Convert CSV annotations → YOLO txt format

def convert_annotations(csv_path: Path, labels_dir: Path, class_map={'Graffiti': 0}):
    """
    Convert CSV bbox annotations to YOLO txt files.
    """
    df = pd.read_csv(csv_path)
    labels_dir.mkdir(parents=True, exist_ok=True)

    for img_name, group in df.groupby('filename'):
        w, h = group[['width','height']].iloc[0]
        lines = []
        for _, row in group.iterrows():
            cid = class_map[row['class']]
            x_c = ((row.xmin + row.xmax) / 2) / w
            y_c = ((row.ymin + row.ymax) / 2) / h
            bw  = (row.xmax - row.xmin) / w
            bh  = (row.ymax - row.ymin) / h
            lines.append(f"{cid} {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}")

        out_file = labels_dir / f"{Path(img_name).stem}.txt"
        out_file.write_text("\n".join(lines))

convert_annotations(BASE/'train_labels.csv', BASE/'labels'/'train')
convert_annotations(BASE/'test_labels.csv',  BASE/'labels'/'test')

In [129]:
# Prepare dataset folder structure + graffiti.yaml

def prepare_dataset(src_img_train: Path,
                    src_img_test:  Path,
                    lbl_train_dir:  Path,
                    lbl_test_dir:   Path,
                    dst:            Path,
                    n_train=400,
                    n_val=40,
                    seed=None):
    """
    Sample and copy images + labels into YOLOv5 structure, then write graffiti.yaml.
    """
    random.seed(seed)
    dst_img_tr = dst/'images'/'train'
    dst_img_va = dst/'images'/'val'
    dst_lbl_tr = dst/'labels'/'train'
    dst_lbl_va = dst/'labels'/'val'
    for p in (dst_img_tr, dst_img_va, dst_lbl_tr, dst_lbl_va):
        p.mkdir(parents=True, exist_ok=True)

    all_train = list(src_img_train.glob('*.jpg'))
    all_test  = list(src_img_test.glob('*.jpg'))

    # adjust sample size to available images to avoid ValueError
    if n_train > len(all_train):
        print(f"⚠️ Requested n_train={n_train} but only {len(all_train)} train images available. Using all available.")
        n_train = len(all_train)
    if n_val > len(all_test):
        print(f"⚠️ Requested n_val={n_val} but only {len(all_test)} test images available. Using all available.")
        n_val = len(all_test)

    if n_train == 0 or n_val == 0:
        raise ValueError(f"Not enough images to sample: n_train={n_train}, n_val={n_val}. Check your source image folders.")

    train_sel = random.sample(all_train, n_train)
    val_sel   = random.sample(all_test,  n_val)

    # copy images and corresponding .txt labels
    for imgs, dst_img, src_lbl, dst_lbl in [
        (train_sel, dst_img_tr, lbl_train_dir, dst_lbl_tr),
        (val_sel,   dst_img_va, lbl_test_dir,  dst_lbl_va)
    ]:
        for img in imgs:
            shutil.copy(img, dst_img/img.name)
            txt = src_lbl / f"{img.stem}.txt"
            if txt.exists():
                shutil.copy(txt, dst_lbl/f"{img.stem}.txt")

    # Write dataset YAML
    yaml_dict = {
        'path': str(dst),
        'train': 'images/train',
        'val':   'images/val',
        'nc':    1,
        'names': ['Graffiti']
    }
    with open(dst/'graffiti.yaml','w') as f:
        yaml.dump(yaml_dict, f)

In [92]:
def evaluate_model(model, img_paths, lbl_dir, save_csv):
    rows = []
    for img_path in img_paths:
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Warning: Could not load image {img_path}")
            continue
            
        h, w = img.shape[:2]
        
        try:
            preds = model(img).xyxy[0].cpu().numpy()
        except Exception as e:
            print(f"Warning: Model prediction failed for {img_path}: {e}")
            preds = np.array([])
        
        gts = load_ground_truth(lbl_dir/f"{img_path.stem}.txt", w, h)

        if len(preds) == 0:
            rows.append([img_path.name, 0.0, 0.0])
        else:
            best = preds[preds[:,4].argmax()]
            x1, y1, x2, y2, conf, _ = best
            ious = [compute_iou([x1,y1,x2,y2], g) for g in gts]
            rows.append([img_path.name, float(conf), max(ious) if ious else 0.0])

    if not rows:
        print("Warning: No evaluation rows generated!")
        return 0.0
    
    df = pd.DataFrame(rows, columns=['image_name','confidence','IoU'])
    Path(save_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_csv, index=False)
    return (df['IoU']>=0.9).mean()


In [118]:
# Helper functions for training & evaluation

def train_yolo(data_yaml: str,
               weights:    str,
               exp_name:   str,
               epochs=30,
               img_size=640,
               batch=16,
               cache=True):
    """
    Runs yolov5/train.py inside YOLO_DIR.
    """
    cmd = [
        'python', 'train.py',
        '--img',   str(img_size),
        '--batch', str(batch),
        '--epochs',str(epochs),
        '--data',  data_yaml,
        '--weights', weights,
        '--name',  exp_name
    ] + (['--cache'] if cache else [])

    subprocess.run(cmd,
                   cwd=str(YOLO_DIR),
                   check=True)

def load_ground_truth(txt_path: Path, img_w: int, img_h: int):
    boxes = []
    if not txt_path.exists():
        return boxes
    for line in txt_path.read_text().splitlines():
        _, x_c, y_c, w, h = map(float, line.split())
        x1 = (x_c - w/2) * img_w
        y1 = (y_c - h/2) * img_h
        x2 = (x_c + w/2) * img_w
        y2 = (y_c + h/2) * img_h
        boxes.append([x1,y1,x2,y2])
    return boxes

def compute_iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter = max(0, xB-xA)*max(0, yB-yA)
    areaA = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)

def evaluate_model(model, img_paths, lbl_dir, save_csv):
    rows = []
    for img_path in img_paths:
        img = cv2.imread(str(img_path))
        h,w = img.shape[:2]
        preds = model(img).xyxy[0].cpu().numpy()
        gts   = load_ground_truth(lbl_dir/f"{img_path.stem}.txt", w, h)

        if len(preds)==0:
            rows.append([img_path.name, 0.0, 0.0])
        else:
            best = preds[preds[:,4].argmax()]
            x1,y1,x2,y2,conf,_ = best
            ious = [compute_iou([x1,y1,x2,y2], g) for g in gts]
            rows.append([img_path.name, float(conf), max(ious) if ious else 0.0])

    df = pd.DataFrame(rows, columns=['image_name','confidence','IoU'])
    df.to_csv(save_csv, index=False)
    return (df['IoU']>=0.9).mean()

In [ ]:
# Enhanced training function that caches and explicitly saves each best.pt

import shutil

def train_yolo_and_save(data_yaml: str,
                        weights:    str,
                        exp_name:   str,
                        epochs=30,
                        img_size=640,
                        batch=16,
                        cache=True):
    """
    Runs yolov5/train.py with caching, then copies best.pt to yolov5/models/exp_name.pt
    """
    cmd = [
        'python', 'train.py',
        '--img',    str(img_size),
        '--batch',  str(batch),
        '--epochs', str(epochs),
        '--data',   data_yaml,
        '--weights',weights,
        '--name',   exp_name
    ] + (['--cache'] if cache else [])

    # 1) launch training
    subprocess.run(cmd,
                   cwd=str(YOLO_DIR),
                   check=True)

    # 2) locate the best weights
    src = YOLO_DIR / 'runs' / 'train' / exp_name / 'weights' / 'best.pt'
    
    # 3) copy to a central models folder
    models_dir = YOLO_DIR / 'models'
    models_dir.mkdir(parents=True, exist_ok=True)
    dst = models_dir / f"{exp_name}.pt"
    shutil.copy(src, dst)
    print(f"Saved best.pt : {dst}")

    return str(dst)

In [130]:
# %% Cell X: full iterative training+evaluation

data_yaml     = str(BASE/'dataset'/'graffiti.yaml')
prev_weights  = 'yolov5s.pt'   # initial pretrained
max_iters     = 10

for i in range(1, max_iters+1):
    # 1) re-sample into dataset/images/train & val
    prepare_dataset(
        src_img_train=BASE/'images'/'train',
        src_img_test= BASE/'images'/'test',
        lbl_train_dir=BASE/'labels'/'train',
        lbl_test_dir= BASE/'labels'/'test',
        dst=BASE/'dataset',
        n_train=400,
        n_val=40,
        seed=None               # different sample each time
    )
    
    # 2) train and save best.pt
    exp_name     = f'graffiti_exp{i}'
    best_weights = train_yolo_and_save(data_yaml, prev_weights, exp_name, epochs=30)
    
    # 3) evaluate on this iteration’s 40
    val_imgs     = list((BASE/'dataset'/'images'/'val').glob('*.jpg'))
    rate         = evaluate_model(
                       torch.hub.load('ultralytics/yolov5','custom',
                                      path=best_weights, force_reload=False).to(DEVICE),
                       val_imgs,
                       BASE/'dataset'/'labels'/'val',
                       save_csv=f'Iteration_Results/iteration{i}_results.csv'
                   )
    print(f"Iteration {i}: {rate*100:.1f}% ≥ 0.9 IoU")
    
    # 4) break if success
    if rate >= 0.8:
        print("Reached 80% of images ≥ 90% IoU; stopping.")
        break
    
    # prepare for next iteration
    prev_weights = best_weights

⚠️ Requested n_train=400 but only 10 train images available. Using all available.
⚠️ Requested n_val=40 but only 5 test images available. Using all available.
Using batch size: 4
Running training: graffiti_exp1
Weights: yolov5s.pt
Warnings/Errors (last 10 lines):
   This initial message can be silenced or aggravated in the future by setting the
   $GIT_PYTHON_REFRESH environment variable. Use one of the following values:
       - quiet|q|silence|s|silent|none|n|0: for no message or exception
       - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
       - error|e|exception|raise|r|2: for a raised exception
   Example:
       export GIT_PYTHON_REFRESH=quiet
⚠️ Training returned code 1, using original weights as fallback


Using cache found in C:\Users\Admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-3-22 Python-3.11.9 torch-2.10.0+cpu CPU

100%|██████████| 14.1M/14.1M [00:01<00:00, 9.57MB/s]

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


Iteration 1: 0.0% ≥ 0.9 IoU
⚠️ Requested n_train=400 but only 10 train images available. Using all available.
⚠️ Requested n_val=40 but only 5 test images available. Using all available.
Using batch size: 4
Running training: graffiti_exp2
Weights: yolov5s.pt


Using cache found in C:\Users\Admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-3-22 Python-3.11.9 torch-2.10.0+cpu CPU

Fusing layers... 


Warnings/Errors (last 10 lines):
   This initial message can be silenced or aggravated in the future by setting the
   $GIT_PYTHON_REFRESH environment variable. Use one of the following values:
       - quiet|q|silence|s|silent|none|n|0: for no message or exception
       - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
       - error|e|exception|raise|r|2: for a raised exception
   Example:
       export GIT_PYTHON_REFRESH=quiet
⚠️ Training returned code 1, using original weights as fallback


...

Iteration 9: 0.0% ≥ 0.9 IoU
⚠️ Requested n_train=400 but only 10 train images available. Using all available.
⚠️ Requested n_val=40 but only 5 test images available. Using all available.
Using batch size: 4
Running training: graffiti_exp10
Weights: yolov5s.pt


Using cache found in C:\Users\Admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-3-22 Python-3.11.9 torch-2.10.0+cpu CPU

Fusing layers... 


Warnings/Errors (last 10 lines):
   This initial message can be silenced or aggravated in the future by setting the
   $GIT_PYTHON_REFRESH environment variable. Use one of the following values:
       - quiet|q|silence|s|silent|none|n|0: for no message or exception
       - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
       - error|e|exception|raise|r|2: for a raised exception
   Example:
       export GIT_PYTHON_REFRESH=quiet
⚠️ Training returned code 1, using original weights as fallback


YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


Iteration 10: 0.0% ≥ 0.9 IoU


In [131]:
# Setup: Download YOLOv5 repo + cache pre-trained weights locally

import urllib.request
import zipfile
import ssl

def download_file_with_ssl_bypass(url, output_path):
    """Download file with SSL verification disabled."""
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE
    
    try:
        # Install SSL context as default
        https_handler = urllib.request.HTTPSHandler(context=ssl_context)
        opener = urllib.request.build_opener(https_handler)
        urllib.request.install_opener(opener)
        
        # Now download
        urllib.request.urlretrieve(url, output_path)
        return True
    except Exception as e:
        print(f"Download failed: {e}")
        return False

if not YOLO_DIR.exists():
    print(f"Downloading YOLOv5 to {YOLO_DIR}...")
    YOLO_DIR.parent.mkdir(parents=True, exist_ok=True)
    
    zip_path = YOLO_DIR.parent / 'yolov5.zip'
    print("Downloading YOLOv5 repo from GitHub...")
    if not download_file_with_ssl_bypass(
        'https://github.com/ultralytics/yolov5/archive/master.zip',
        str(zip_path)
    ):
        raise RuntimeError("Failed to download YOLOv5")
    
    print("Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(str(YOLO_DIR.parent))
    
    extracted = YOLO_DIR.parent / 'yolov5-master'
    if extracted.exists():
        extracted.replace(YOLO_DIR)
    
    zip_path.unlink()
    print("YOLOv5 extracted successfully!")
    
    # Install requirements
    print("Installing YOLOv5 requirements...")
    result = subprocess.run(
        ['pip', 'install', '-r', str(YOLO_DIR / 'requirements.txt')],
        capture_output=True,
        text=True,
        timeout=120
    )
    if result.returncode != 0:
        print("Warning: Some requirements may not have installed properly")
else:
    print(f"YOLOv5 already exists at {YOLO_DIR}")

# Pre-download and cache YOLOv5 model weights
weights_dir = YOLO_DIR / 'weights'
weights_dir.mkdir(parents=True, exist_ok=True)

weights_file = weights_dir / 'yolov5s.pt'
if not weights_file.exists():
    print(f"\nDownloading pre-trained YOLOv5s weights...")
    # Try to download from Ultralytics release
    weights_urls = [
        'https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt',
        'https://github.com/ultralytics/yolov5/releases/download/v6.2/yolov5s.pt',
    ]
    
    downloaded = False
    for url in weights_urls:
        print(f"Trying: {url}")
        if download_file_with_ssl_bypass(url, str(weights_file)):
            print(f"Successfully downloaded to {weights_file}")
            downloaded = True
            break
    
    if not downloaded:
        print("Warning: Could not download pre-trained weights. Will use yolov5s.pt from hub on first training run.")
else:
    print(f"Weights already cached at {weights_file}")

print("Setup complete!")


YOLOv5 already exists at D:\COS40007\Week_06\yolov5
Weights already cached at D:\COS40007\Week_06\yolov5\weights\yolov5s.pt
Setup complete!


In [132]:
# %% Cell 10: Organise CSV + pick two “good” samples per iteration

import pandas as pd
import shutil
from pathlib import Path
import cv2

def collect_iteration_outputs(iteration,
                              model,
                              dataset_base,
                              results_folder='outputs'):
    """
    - Moves iteration{i}_results.csv → outputs/iteration{i}/
    - Picks top-2 by IoU, runs inference & writes their rendered images
      into outputs/iteration{i}/samples/
    """
    # paths
    csv_src    = Path(f'Iteration_Results/iteration{iteration}_results.csv')
    out_iter   = Path(results_folder)/f'iteration{iteration}'
    samples_dir= out_iter/'samples'
    val_imgs   = dataset_base/'images'/'val'
    lbl_dir    = dataset_base/'labels'/'val'
    
    # make directories
    samples_dir.mkdir(parents=True, exist_ok=True)
    
    # 1) move CSV
    csv_dst = out_iter/csv_src.name
    if not csv_src.exists():
        print(f"⚠️ Skipping iteration {iteration} output collection because CSV not found: {csv_src}")
        return

    out_iter.mkdir(parents=True, exist_ok=True)
    shutil.move(str(csv_src), str(csv_dst))
    
    # 2) pick top 2 by IoU
    df = pd.read_csv(csv_dst)
    top2 = df.sort_values('IoU', ascending=False).head(10)['image_name'].tolist()
    
    # 3) for each top image: run model & save rendered image
    for img_name in top2:
        img_path = val_imgs/img_name
        img      = cv2.imread(str(img_path))
        res      = model(img)             # yolo inference
        rendered = res.render()[0]        # BGR array
        
        out_path = samples_dir/img_name
        cv2.imwrite(str(out_path), rendered)
        print(f"Saved sample : {out_path}")


def resolve_model_for_iteration(i):
    candidate = YOLO_DIR/'models'/f'graffiti_exp{i}.pt'
    if candidate.exists():
        return candidate

    candidates = sorted((YOLO_DIR/'models').glob('graffiti_exp*.pt'), key=lambda p: p.stat().st_mtime)
    if candidates:
        print(f"⚠️ Iteration model not found ({candidate}), using latest available model {candidates[-1]}")
        return candidates[-1]

    fallback = YOLO_DIR/'weights'/'yolov5s.pt'
    if fallback.exists():
        print(f"⚠️ Iteration model not found; falling back to pretrained {fallback}")
        return fallback

    raise FileNotFoundError(f"No valid model weights found for iteration {i}")

for i in range(1, max_iters+1):    # achieved_iter is the last iteration you ran
    weights = resolve_model_for_iteration(i)

    try:
        model = torch.hub.load('ultralytics/yolov5','custom', path=str(weights), force_reload=False).to(DEVICE)
    except Exception as e:
        print(f"❌ Could not load model for iteration {i} from {weights}: {e}")
        continue

    collect_iteration_outputs(i,
                               model,
                               BASE/'dataset',
                               results_folder='outputs')

⚠️ Iteration model not found; falling back to pretrained D:\COS40007\Week_06\yolov5\weights\yolov5s.pt


Using cache found in C:\Users\Admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-3-22 Python-3.11.9 torch-2.10.0+cpu CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


Saved sample : outputs\iteration1\samples\dummy_000.jpg
Saved sample : outputs\iteration1\samples\dummy_001.jpg
Saved sample : outputs\iteration1\samples\dummy_002.jpg
Saved sample : outputs\iteration1\samples\dummy_003.jpg


Using cache found in C:\Users\Admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-3-22 Python-3.11.9 torch-2.10.0+cpu CPU

Fusing layers... 


Saved sample : outputs\iteration1\samples\dummy_004.jpg
⚠️ Iteration model not found; falling back to pretrained D:\COS40007\Week_06\yolov5\weights\yolov5s.pt


...

Saved sample : outputs\iteration9\samples\dummy_000.jpg
Saved sample : outputs\iteration9\samples\dummy_001.jpg
Saved sample : outputs\iteration9\samples\dummy_002.jpg
Saved sample : outputs\iteration9\samples\dummy_003.jpg


Using cache found in C:\Users\Admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-3-22 Python-3.11.9 torch-2.10.0+cpu CPU

Fusing layers... 


Saved sample : outputs\iteration9\samples\dummy_004.jpg
⚠️ Iteration model not found; falling back to pretrained D:\COS40007\Week_06\yolov5\weights\yolov5s.pt


YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


Saved sample : outputs\iteration10\samples\dummy_000.jpg
Saved sample : outputs\iteration10\samples\dummy_001.jpg
Saved sample : outputs\iteration10\samples\dummy_002.jpg
Saved sample : outputs\iteration10\samples\dummy_003.jpg
Saved sample : outputs\iteration10\samples\dummy_004.jpg


In [133]:
#  Visualize detections on a set of images inline

import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path

def visualize_batch(model, image_paths, out_folder="vis_outputs"):
    """
    Runs model on each image, saves rendered results to out_folder,
    then displays them inline.
    """
    out_folder = Path(out_folder)
    out_folder.mkdir(exist_ok=True)
    
    for img_path in image_paths:
        img = cv2.imread(str(img_path))
        res = model(img)
        rendered = res.render()[0]               # numpy array with boxes drawn
        save_to = out_folder / img_path.name
        cv2.imwrite(str(save_to), rendered)
        
        # convert BGR→RGB for display
        rgb = cv2.cvtColor(rendered, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(6,6))
        plt.imshow(rgb)
        plt.axis('off')
        display(plt.gcf())
        plt.close()

final_model = find_best_visualization_model()
try:
    viz_model = torch.hub.load('ultralytics/yolov5','custom', path=str(final_model), force_reload=False).to(DEVICE)
    imgs = list((BASE/'dataset'/'images'/'val').glob('*.jpg'))
    if not imgs:
        print("⚠️ No validation images found; visualization skipped.")
    else:
        visualize_batch(viz_model, imgs[:40])    # show first 40
except Exception as e:
    print(f"❌ Visualization model load failed: {e}")

Using cache found in C:\Users\Admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-3-22 Python-3.11.9 torch-2.10.0+cpu CPU

Fusing layers... 


⚠️ No expo model found; falling back to pretrained D:\COS40007\Week_06\yolov5\weights\yolov5s.pt


YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


<Figure size 600x600 with 1 Axes>

<Figure size 600x600 with 1 Axes>

<Figure size 600x600 with 1 Axes>

<Figure size 600x600 with 1 Axes>

<Figure size 600x600 with 1 Axes>

In [135]:
# Single-video inference

def resolve_inference_model():
    """Pick the best available trained model, then fallback to yolov5s.pt if needed."""
    models_dir = YOLO_DIR / 'models'
    candidates = sorted(models_dir.glob('graffiti_exp*.pt'), key=lambda p: p.stat().st_mtime)
    if candidates:
        print(f"Using trained model for inference: {candidates[-1]}")
        return candidates[-1]

    fallback = YOLO_DIR / 'weights' / 'yolov5s.pt'
    if fallback.exists():
        print(f"Fallback to pretrained weights: {fallback}")
        return fallback

    raise FileNotFoundError("No model weights are available for video inference.")


def process_video(video_path, weights_path, output_path, class_names=['Graffiti']):
    """
    Runs YOLOv5 on each frame of video_path, writes annotated video to output_path.
    """
    if not Path(weights_path).exists():
        raise FileNotFoundError(f"Weights file not found: {weights_path}")

    model = torch.hub.load('ultralytics/yolov5','custom', path=weights_path, force_reload=False).to(DEVICE)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"⚠️ Skipping video because it cannot be opened: {video_path}")
        return

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    print(f"Processing {video_path} → {output_path}")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        results = model(frame)
        frame   = results.render()[0]
        out.write(frame)

    cap.release()
    out.release()
    print(f"Done: {output_path}")


# Process all available videos
model_for_videos = resolve_inference_model()
for idx in range(1, 6):
    input_vid  = BASE/'videos'/f'{idx}.mp4'
    output_vid = BASE/'videos'/f'{idx}_detected.mp4'
    try:
        process_video(input_vid, str(model_for_videos), output_vid)
    except Exception as e:
        print(f"❌ Video {input_vid} failed: {e}")
        continue

Using cache found in C:\Users\Admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-3-22 Python-3.11.9 torch-2.10.0+cpu CPU

Fusing layers... 


Fallback to pretrained weights: D:\COS40007\Week_06\yolov5\weights\yolov5s.pt


YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 
Using cache found in C:\Users\Admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-3-22 Python-3.11.9 torch-2.10.0+cpu CPU

Fusing layers... 


⚠️ Skipping video because it cannot be opened: D:\COS40007\Week_06\videos\1.mp4


YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 
Using cache found in C:\Users\Admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-3-22 Python-3.11.9 torch-2.10.0+cpu CPU



...

⚠️ Skipping video because it cannot be opened: D:\COS40007\Week_06\videos\3.mp4


YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 
Using cache found in C:\Users\Admin/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-3-22 Python-3.11.9 torch-2.10.0+cpu CPU

Fusing layers... 


⚠️ Skipping video because it cannot be opened: D:\COS40007\Week_06\videos\4.mp4


YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


⚠️ Skipping video because it cannot be opened: D:\COS40007\Week_06\videos\5.mp4
